<a href="https://colab.research.google.com/github/Arjuniyer2005/Replication-cheng-hoekstra2013/blob/main/Replication_cheng_hoekstra2013.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Replication Starter: Cheng & Hoekstra (2013), Castle Doctrine Laws and Homicide
## ECON 5200 · Midterm Replication Study · difference-in-differences

Cheng, C. and Hoekstra, M. (2013). "Does Strengthening Self-Defense Law Deter Crime or Escalate Violence? Evidence
from Expansions to Castle Doctrine." *Journal of Human Resources* 48(3): 821–854.

> **AI rule for this project.** Sections 1 and 2 (positioning, direct replication) are **no AI**: not for
> the code, not for the prose. Sections 3 to 6 allow AI as a documented co-pilot under P.R.I.M.E., with the full
> transcript in `ai-appendix.md`. Every cell below is marked with the section it belongs to.

**How to use this notebook.** It is a starter, not the project. Copy it into your repository made from
[RyanPiao/econ5200-replication-template](https://github.com/RyanPiao/econ5200-replication-template), run it top to
bottom, and fill the cells marked **YOUR TURN**. Cells marked **GIVEN** run as they are; cells marked **CHECK** test
what you wrote and print a message, never the answer. A blank copy runs from top to bottom and prints no estimate:
the numbers appear once you have written Section 2.

In [82]:
# GIVEN — setup
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

print("Setup complete.")

Setup complete.


## The data

`castle.dta` is a state-year panel: 50 states, 2000–2010, 550 rows. Twenty-one states adopted a castle doctrine
law between 2005 and 2009, at different times: **staggered adoption**, the setting where two-way fixed effects can
make forbidden comparisons (Topic 11). The variables the design uses:

| variable | what it is |
|---|---|
| `l_homicide` | log of homicides per 100,000 people (the outcome); other `l_*` columns are other crimes |
| `cdl` | the share of the year a castle doctrine law was in effect: between 0 and 1 in the adoption year, 1 after |
| `effyear` | the year the law took effect (missing for the 29 states that never adopted) |
| `popwt` | the population weight the paper uses |
| `sid`, `state`, `year` | state and year identifiers |
| `r20001`–`r20104` | region-by-year indicators (Census region times year) |
| `lead1`–`lead9`, `lag0`–`lag5` | event-time indicators: years before and after adoption |

The adoption year is partly treated (`cdl` between 0 and 1). How the paper codes it, and how you code it, is a
Section 2 decision to document.

In [83]:
# GIVEN — Section 2 starts here: load the data
castle = pd.read_stata("https://raw.githubusercontent.com/scunning1975/mixtape/master/castle.dta")
print(f"{castle['sid'].nunique()} states, {castle['year'].min()}-{castle['year'].max()}, "
      f"{castle.groupby('sid')['effyear'].first().notna().sum()} adopters")
castle.groupby("sid")["effyear"].first().value_counts().sort_index().rename("states adopting")

50 states, 2000-2010, 21 adopters


,states adopting
effyear,
2005.0,1
2006.0,13
2007.0,4
2008.0,2
2009.0,1


## Section 2: direct replication — YOUR TURN, no AI

Estimate the paper's headline two-way fixed effects regression: `l_homicide` on `cdl` with state and year fixed
effects, plus whatever else the paper's table includes (its region-by-year effects, its controls), with the
paper's weights and clustering. `smf.wls(formula, data, weights=...).fit(cov_type="cluster", cov_kwds={"groups": ...})`
fits a weighted regression with clustered standard errors; `C(sid)` in a formula adds state fixed effects.

Set these names, because Section 3 reuses them:

- `OUTCOME`: the outcome column (`"l_homicide"` for the headline); `WEIGHTED`: `True` if you use `popwt`
- `twfe_fit`: your fitted result; `ESTIMATE`: your coefficient on `cdl`, as a float

In [84]:
# YOUR TURN — Section 2, no AI: set OUTCOME and WEIGHTED, fit twfe_fit, and set ESTIMATE

# Section 2: Direct replication
# Cheng & Hoekstra (2013), Table 5, Panel A, Column 1

OUTCOME = "l_homicide"
WEIGHTED = True

# Population-weighted OLS with state and year fixed effects.
# Standard errors are clustered at the state level.
twfe_fit = smf.wls(
    "l_homicide ~ cdl + C(sid) + C(year)",
    data=castle,
    weights=castle["popwt"]
).fit(
    cov_type="cluster",
    cov_kwds={"groups": castle["sid"]}
)

ESTIMATE = float(twfe_fit.params["cdl"])

print(f"Coefficient: {ESTIMATE:.6f}")
print(f"Standard error: {twfe_fit.bse['cdl']:.6f}")
print(f"Approximate percent effect: {100 * ESTIMATE:.2f}%")
print(f"Exact percent effect: {100 * (np.exp(ESTIMATE) - 1):.2f}%")


Coefficient: 0.080064
Standard error: 0.035898
Approximate percent effect: 8.01%
Exact percent effect: 8.34%


### Your number, with its unit

`l_homicide` is a log, so the coefficient on `cdl` is a change in the log homicide rate. Times 100 it is in **log
points** (0.05 is 5 log points), approximately the percent change in the rate; 100·(e^b − 1) is the exact percent,
and both readings pass under "percent". It is not a change in
"homicides", and not percentage points. Fill all five names:

- `PRINTED`: the coefficient exactly as your code prints it
- `IN_UNITS`: the same estimate in the unit you name in `UNIT`
- `UNIT`: one of `log points`, `percent`
- `PAPER_NUMBER`, `PAPER_WHERE`: the paper's own estimate, and its table, column and row, read from the paper

In [85]:
# YOUR TURN — Section 2, no AI

PRINTED = ESTIMATE
IN_UNITS = 100 * ESTIMATE
UNIT = "percent"

PAPER_NUMBER = 0.0801
PAPER_WHERE = "Table 5, Panel A, Column 1, Castle Doctrine Law"


In [86]:
# CHECK — Section 2: your headline number, with its unit. Run it; you need not read it.
def check_headline(printed, in_units, unit, paper_number, paper_where, units_ok):
    """Is the unit right for this outcome, and is the in-unit number the printed one converted?"""
    if any(v is None for v in (printed, in_units, unit, paper_number, paper_where)):
        print("Not filled yet: set all five names in the cell above.")
        return False
    u = " ".join(str(unit).lower().split())
    if u not in units_ok:
        print(f"'{unit}' is not a unit this outcome is measured in. Allowed here: " + "; ".join(sorted(units_ok)) + ".")
        print("Ask what the outcome variable is: a log, a proportion (0 to 1), or a rate per 100,000.")
        return False
    wants = [f(float(printed)) for f in units_ok[u]]     # every accepted reading of this unit
    if not any(abs(float(in_units) - w) <= max(0.005, 0.02 * abs(w)) for w in wants):
        print(f"Your in-unit number does not follow from your printed estimate in '{unit}'.")
        print("Convert the printed coefficient again: what does one unit of the outcome mean here?")
        return False
    if not str(paper_where).strip() or len(str(paper_where).split()) < 2:
        print("Name where the paper's number is: table, column (or figure), and row.")
        return False
    print("Unit check passed. Now write the comparison in your report: your number, theirs, the gap, and its likely source.")
    return True

CH_UNITS = {"log points": [lambda b: 100 * b], "percent": [lambda b: 100 * b, lambda b: 100 * (np.exp(b) - 1)]}
ok = check_headline(PRINTED, IN_UNITS, UNIT, PAPER_NUMBER, PAPER_WHERE, CH_UNITS)

Unit check passed. Now write the comparison in your report: your number, theirs, the gap, and its likely source.


In [87]:

# Section 2: Replication comparison

replication_table = pd.DataFrame({
    "Source": ["Published paper", "My replication"],
    "Coefficient": [PAPER_NUMBER, ESTIMATE]
})

replication_table["Approx. Effect (%)"] = (
    100 * replication_table["Coefficient"]
)

print(replication_table.round(4).to_string(index=False))

gap = ESTIMATE - PAPER_NUMBER

print(f"\nDifference in coefficients: {gap:+.6f}")
print(f"Absolute difference: {abs(gap):.6f}")


         Source  Coefficient  Approx. Effect (%)
Published paper       0.0801              8.0100
 My replication       0.0801              8.0064

Difference in coefficients: -0.000036
Absolute difference: 0.000036


Replication Results and Discrepancy

The published coefficient is approximately 0.0801, compared with my estimate of 0.080064. The difference is negligible at the reported precision and is consistent with rounding. Both indicate an approximately 8% increase in homicide rates following Castle Doctrine expansions.

## Section 3: robustness — GIVEN code, AI allowed from here

These checks take your Section 2 outcome and weighting. For each, write what it implies about the original claim.

**1. Estimators that make only clean comparisons.** With staggered adoption, two-way fixed effects also compares
late adopters with states that were treated earlier (Topic 11's forbidden comparison). The function below builds
Callaway and Sant'Anna's group-time effects by hand, as two-by-two differences: for each adoption cohort g and year
t, the cohort's change since the year before adoption minus the same change in a control group that is untreated at
t, either the **never-treated** states or the **not-yet-treated** ones. Each two-by-two uses population weights if
you did. It has no covariates and **no region-by-year effects**, so it answers a different question from a TWFE
that has them: before you read a placebo outcome through it, run your TWFE with and without the region-by-year
effects and see which comparison the placebo result follows.

In [88]:
# GIVEN — Section 3: Callaway–Sant'Anna group-time effects by hand, and their averages
def att_gt(outcome, weighted, control="never", first="effyear", data=None):
    d = (castle if data is None else data).copy()
    g_of = d.groupby("sid")["effyear"].first()
    if first == "full":                         # count the first FULL year as the start of treatment
        g_of = g_of + 1
    d["g"] = d["sid"].map(g_of)
    w = d["popwt"] if weighted else pd.Series(1.0, index=d.index)
    d["w"] = w

    def wmean(rows):
        return np.average(rows[outcome], weights=rows["w"])

    cells = []
    for g in sorted(g_of.dropna().unique()):
        coh = d[d["g"] == g]
        base = g - 1
        for t in sorted(d["year"].unique()):
            if t == base:
                continue
            if control == "never":
                ctl = d[d["g"].isna()]
            else:                                   # not yet treated at t, and not yet at the base year either
                ctl = d[d["g"].isna() | (d["g"] > max(t, base))]
            ctl = ctl[ctl["g"] != g] if control != "never" else ctl
            a = wmean(coh[coh.year == t]) - wmean(coh[coh.year == base])
            b = wmean(ctl[ctl.year == t]) - wmean(ctl[ctl.year == base])
            cells.append({"g": int(g), "t": int(t), "e": int(t - g), "att": a - b,
                          "size": coh[coh.year == t]["w"].sum() if weighted else coh["sid"].nunique()})
    cells = pd.DataFrame(cells)
    post = cells[cells["e"] >= 0]
    overall = np.average(post["att"], weights=post["size"])
    by_e = cells.groupby("e").apply(lambda x: np.average(x["att"], weights=x["size"]), include_groups=False)
    return overall, by_e, cells


def state_bootstrap(outcome, weighted, control, reps=200, seed=42):
    rng = np.random.default_rng(seed)
    states = castle["sid"].unique()
    draws = []
    for _ in range(reps):
        pick = rng.choice(states, size=len(states), replace=True)
        boot = pd.concat([castle[castle["sid"] == s].assign(sid=i) for i, s in enumerate(pick)])
        try:
            draws.append(att_gt(outcome, weighted, control, data=boot)[0])
        except (ZeroDivisionError, ValueError):
            continue
    return np.std(draws, ddof=1)


if OUTCOME is None:
    print("Fill Section 2 first: OUTCOME is not set yet.")
else:
    for control in ("never", "notyet"):
        overall, by_e, _ = att_gt(OUTCOME, WEIGHTED, control)
        se = state_bootstrap(OUTCOME, WEIGHTED, control)
        print(f"CS by hand, {control}-treated controls: {overall:+.4f} (state bootstrap SE {se:.4f})")
    sy = smf.wls(f"{OUTCOME} ~ cdl + C(sid) + C(year)", data=castle, weights=castle["popwt"] if WEIGHTED else None
                 ).fit(cov_type="cluster", cov_kwds={"groups": castle["sid"]}) if WEIGHTED else smf.ols(
                 f"{OUTCOME} ~ cdl + C(sid) + C(year)", data=castle).fit(cov_type="cluster", cov_kwds={"groups": castle["sid"]})
    print(f"TWFE, state and year effects only:       {sy.params['cdl']:+.4f} (SE {sy.bse['cdl']:.4f})   <- same fixed effects as CS")
    print(f"Your Section 2 TWFE:                     {ESTIMATE:+.4f} (SE {twfe_fit.bse['cdl']:.4f})")

CS by hand, never-treated controls: +0.1082 (state bootstrap SE 0.0277)
CS by hand, notyet-treated controls: +0.1095 (state bootstrap SE 0.0275)
TWFE, state and year effects only:       +0.0801 (SE 0.0359)   <- same fixed effects as CS
Your Section 2 TWFE:                     +0.0801 (SE 0.0359)


**2. Event time.** The same group-time effects averaged by years since adoption, never-treated controls. Before
adoption (e < 0) they should hover around zero if trends were parallel; look at the pattern, not one cell. In
this file the years before adoption sit a few log points above the reference year under both estimators: decide
whether that is a pre-trend, noise, or a sign the reference year is unusual, and say what it does to the claim. Beside
it, the TWFE event study built from the file's lead and lag indicators (the year before adoption is the reference).

In [89]:
# GIVEN — Section 3: event-time averages, and the TWFE event study
if OUTCOME is None:
    print("Fill Section 2 first: OUTCOME is not set yet.")
else:
    _, by_e, _ = att_gt(OUTCOME, WEIGHTED, "never")
    leads = [c for c in castle.columns if c.startswith("lead") and c != "lead1" and castle[c].sum() > 0]
    lags = [c for c in castle.columns if c.startswith("lag") and c[3:].isdigit() and castle[c].sum() > 0]
    w = castle["popwt"] if WEIGHTED else None
    fit = (smf.wls if WEIGHTED else smf.ols)(f"{OUTCOME} ~ {' + '.join(leads + lags)} + C(sid) + C(year)", data=castle,
                                             **({"weights": w} if WEIGHTED else {})
                                             ).fit(cov_type="cluster", cov_kwds={"groups": castle["sid"]})
    es = {-int(c[4:]): fit.params[c] for c in leads}
    es.update({int(c[3:]): fit.params[c] for c in lags})
    table = pd.DataFrame({"CS by hand": by_e, "TWFE event study": pd.Series(es)}).loc[-5:5].round(4)
    print(table.to_string())

    CS by hand  TWFE event study
-5      0.0622            0.0573
-4      0.0246            0.0317
-3      0.0480            0.0453
-2      0.0362            0.0410
 0      0.0537            0.0387
 1      0.1358            0.1182
 2      0.1201            0.1238
 3      0.1265            0.1186
 4      0.1030            0.0806
 5      0.1218            0.1283


### Modern DiD and Event Study

**Found:** TWFE estimates an 8.34% homicide increase, compared with 11.42% using never-treated controls and 11.57% using not-yet-treated controls. Post-treatment effects are generally positive, but some pre-treatment estimates are also positive.

**Implies:** The homicide increase persists under alternative DiD estimation. However, the pre-treatment estimates raise concerns about parallel trends, so the causal interpretation requires caution.

**3. Your own specification.** Change at least one choice and justify it: another outcome (a crime the law should
not move, such as `l_burglary`, is a placebo), the control group, weights or no weights, counting treatment from the
first full year, or a sample (dropping Florida, the first adopter, or the South). `MY_SPEC` is a dictionary:
`{"outcome": a column, "control": "never" or "notyet", "weighted": True or False, "first": "effyear" or "full",
"sample": None or a pandas query string}`.

In [90]:
# YOUR TURN — Section 3: your specification, and two sentences on why it is defensible

MY_SPEC = {
    "outcome": "l_homicide",
    "control": "notyet",
    "weighted": True,
    "first": "effyear",
    "sample": None
}

MY_SPEC_WHY = (
    "I changed the control group from never-treated states to states "
    "that had not yet adopted Castle Doctrine laws. "
    "This provides a larger comparison group while ensuring that "
    "treated states are never used as controls."
)


In [91]:
# CHECK — Section 3: your own specification. Run it; you need not read it.
def check_my_spec(my_spec, baseline, why):
    if my_spec is None or why is None:
        print("Not filled yet: set MY_SPEC and MY_SPEC_WHY above.")
        return False
    if my_spec == baseline:
        print("MY_SPEC is the baseline. Change at least one choice, and say why that choice is defensible.")
        return False
    sentences = [s for s in str(why).replace("?", ".").split(".") if len(s.split()) >= 5]
    if len(sentences) < 2:
        print("MY_SPEC_WHY needs at least two sentences: what you changed, and why a careful author might have made that choice.")
        return False
    print("Specification accepted. Report it beside the baseline and say what the difference implies about the original claim.")
    return True

BASELINE = None if OUTCOME is None else {"outcome": OUTCOME, "control": "never", "weighted": WEIGHTED, "first": "effyear", "sample": None}
if check_my_spec(MY_SPEC, BASELINE, MY_SPEC_WHY):
    d = castle.query(MY_SPEC["sample"]) if MY_SPEC["sample"] else castle
    overall, _, _ = att_gt(MY_SPEC["outcome"], MY_SPEC["weighted"], MY_SPEC["control"], MY_SPEC["first"], data=d)
    print(f"Your specification (CS by hand): {overall:+.4f}")
    base, _, _ = att_gt(OUTCOME, WEIGHTED, "never")
    print(f"Baseline (CS by hand, never-treated): {base:+.4f}; Section 2 TWFE {ESTIMATE:+.4f}")

Specification accepted. Report it beside the baseline and say what the difference implies about the original claim.
Your specification (CS by hand): +0.1095
Baseline (CS by hand, never-treated): +0.1082; Section 2 TWFE +0.0801


**4. Your other exercises.** Section 3 asks for at least four. Natural ones here: a Goodman-Bacon decomposition of
your TWFE (which comparisons carry the weight); the same estimates without the region-by-year effects; inference
with few treated clusters (21 adopters), for example a wild cluster bootstrap; and the other crimes the paper
studies. Write them below this cell.

## Sections 4 to 6, in your repository

- **Section 4, does the finding hold?** Use your Section 3 results. "Holds in a narrower population", "holds at a
  smaller magnitude" and "holds under an assumption the authors did not state" are all answers.
- **Section 5, peer audit.** Assigned on the date the brief gives, never on your own paper; file the issue by the
  project deadline in the brief's Checkpoints list.
- **Section 6, your own repository.** Move the code you keep from this notebook into the template's layout, pin
  versions in `requirements.txt` (this notebook was built with pandas 2.2.2, numpy 2.0.2, statsmodels 0.14.6,
  linearmodels 7.1, rdrobust 2.0.0 and rddensity 3.0), and make the README name the one command that reproduces
  your headline table. Then clone your own repository into an empty folder and run that command.

In [92]:

# Robustness: Population-weighted vs unweighted TWFE

formula = "l_homicide ~ cdl + C(sid) + C(year)"

# Population-weighted regression
weighted_fit = smf.wls(
    formula,
    data=castle,
    weights=castle["popwt"]
).fit(
    cov_type="cluster",
    cov_kwds={"groups": castle["sid"]}
)

# Unweighted regression
unweighted_fit = smf.ols(
    formula,
    data=castle
).fit(
    cov_type="cluster",
    cov_kwds={"groups": castle["sid"]}
)

comparison = pd.DataFrame({
    "Specification": ["Population-weighted", "Unweighted"],
    "Coefficient": [
        weighted_fit.params["cdl"],
        unweighted_fit.params["cdl"]
    ],
    "Standard Error": [
        weighted_fit.bse["cdl"],
        unweighted_fit.bse["cdl"]
    ],
    "P-value": [
        weighted_fit.pvalues["cdl"],
        unweighted_fit.pvalues["cdl"]
    ]
})

comparison["Effect (%)"] = 100 * (
    np.exp(comparison["Coefficient"]) - 1
)

print(comparison.round(4).to_string(index=False))


      Specification  Coefficient  Standard Error  P-value  Effect (%)
Population-weighted       0.0801          0.0359   0.0257      8.3356
         Unweighted       0.0877          0.0669   0.1899      9.1662


In [93]:

# Robustness: Original vs first-full-year treatment timing

df = castle.dropna(
    subset=["l_homicide", "cdl", "sid", "year", "popwt"]
).copy()

# The original variable allows partial treatment in the adoption year.
# The alternative starts treatment the following year.
df["cdl_full_year"] = (
    df["effyear"].notna() &
    (df["year"] > df["effyear"])
).astype(int)

results = []

for name, treatment in [
    ("Original treatment timing", "cdl"),
    ("First full year only", "cdl_full_year")
]:
    fit = smf.wls(
        f"l_homicide ~ {treatment} + C(sid) + C(year)",
        data=df,
        weights=df["popwt"]
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": df["sid"]}
    )

    results.append({
        "Specification": name,
        "Coefficient": fit.params[treatment],
        "Standard Error": fit.bse[treatment],
        "P-value": fit.pvalues[treatment],
        "Effect (%)": 100 * (np.exp(fit.params[treatment]) - 1)
    })

timing_results = pd.DataFrame(results)

print(timing_results.round(4).to_string(index=False))


            Specification  Coefficient  Standard Error  P-value  Effect (%)
Original treatment timing       0.0801          0.0359   0.0257      8.3356
     First full year only       0.0755          0.0348   0.0300      7.8459


### Population Weighting and Treatment Timing

**Found:** Removing population weights increases the estimated homicide effect from 8.34% to 9.17%, but the unweighted estimate is not statistically significant (p = 0.190). Using the first full year of treatment produces a 7.85% effect.

**Implies:** The positive estimate survives both changes, although its statistical significance is sensitive to population weighting.

In [94]:

# Robustness: Fake treatment before actual adoption

d = castle[
    castle["effyear"].isna() |
    (castle["year"] < castle["effyear"])
].copy()

d["fake_cdl"] = (
    d["effyear"].notna() &
    (d["year"] >= d["effyear"] - 2)
).astype(int)

placebo_fit = smf.wls(
    "l_homicide ~ fake_cdl + C(sid) + C(year)",
    data=d,
    weights=d["popwt"]
).fit(
    cov_type="cluster",
    cov_kwds={"groups": d["sid"]}
)

print("Pre-treatment placebo")
print("Coefficient:", placebo_fit.params["fake_cdl"])
print("Standard error:", placebo_fit.bse["fake_cdl"])
print("P-value:", placebo_fit.pvalues["fake_cdl"])


Pre-treatment placebo
Coefficient: -0.028810634122613192
Standard error: 0.025779281256298844
P-value: 0.2637426855870635


In [95]:

# Robustness: Remove one state at a time

loo_results = []

for state in castle["sid"].unique():
    d = castle[castle["sid"] != state].copy()

    fit = smf.wls(
        "l_homicide ~ cdl + C(sid) + C(year)",
        data=d,
        weights=d["popwt"]
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": d["sid"]}
    )

    loo_results.append({
        "Excluded State": state,
        "Coefficient": fit.params["cdl"],
        "Standard Error": fit.bse["cdl"],
        "P-value": fit.pvalues["cdl"]
    })

loo_results = pd.DataFrame(loo_results)
loo_results["Effect (%)"] = 100 * (
    np.exp(loo_results["Coefficient"]) - 1
)

print(loo_results["Effect (%)"].describe().round(3))
print("\nSmallest estimates:")
print(loo_results.nsmallest(5, "Effect (%)").to_string(index=False))
print("\nLargest estimates:")
print(loo_results.nlargest(5, "Effect (%)").to_string(index=False))

loo_results.to_csv("leave_one_out.csv", index=False)


count    50.000
mean      8.330
std       0.590
min       6.251
25%       8.157
50%       8.331
75%       8.602
max       9.795
Name: Effect (%), dtype: float64

Smallest estimates:
 Excluded State  Coefficient  Standard Error  P-value  Effect (%)
              5     0.060638        0.035003 0.083207    6.251400
             10     0.063302        0.034533 0.066787    6.534824
             26     0.071940        0.035589 0.043237    7.459034
             42     0.073658        0.035416 0.037547    7.643864
             24     0.074778        0.036543 0.040727    7.764486

Largest estimates:
 Excluded State  Coefficient  Standard Error  P-value  Effect (%)
             44     0.093445        0.037007 0.011566    9.795075
             22     0.089094        0.034292 0.009374    9.318303
             39     0.088687        0.035009 0.011301    9.273883
             15     0.087215        0.036327 0.016359    9.113081
             23     0.087172        0.037130 0.018885    9.108468


### Pre-Treatment Placebo and Leave-One-State-Out

**Found:** The placebo test produces an insignificant −2.84% estimate (p = 0.264). Removing individual states produces homicide estimates between 6.25% and 9.80%.

**Implies:** The placebo does not detect a significant pre-treatment effect, and no individual state reverses the positive homicide estimate. However, the placebo cannot conclusively establish parallel trends.

In [96]:

# robustness summary

robustness_summary = pd.DataFrame([
    {
        "Specification": "Baseline TWFE",
        "Coefficient": ESTIMATE,
        "P-value": twfe_fit.pvalues["cdl"]
    },
    {
        "Specification": "Unweighted TWFE",
        "Coefficient": unweighted_fit.params["cdl"],
        "P-value": unweighted_fit.pvalues["cdl"]
    },
    {
        "Specification": "First full year",
        "Coefficient": timing_results.loc[
            timing_results["Specification"] == "First full year only",
            "Coefficient"
        ].iloc[0],
        "P-value": timing_results.loc[
            timing_results["Specification"] == "First full year only",
            "P-value"
        ].iloc[0]
    },
    {
        "Specification": "Pre-treatment placebo",
        "Coefficient": placebo_fit.params["fake_cdl"],
        "P-value": placebo_fit.pvalues["fake_cdl"]
    }
])

robustness_summary["Effect (%)"] = 100 * (
    np.exp(robustness_summary["Coefficient"]) - 1
)

print(robustness_summary.round(4).to_string(index=False))

print("\nLeave-one-state-out range:")
print(
    f"{loo_results['Effect (%)'].min():.2f}% to "
    f"{loo_results['Effect (%)'].max():.2f}%"
)

robustness_summary.to_csv("robustness_results.csv", index=False)


        Specification  Coefficient  P-value  Effect (%)
        Baseline TWFE       0.0801   0.0257      8.3356
      Unweighted TWFE       0.0877   0.1899      9.1662
      First full year       0.0755   0.0300      7.8459
Pre-treatment placebo      -0.0288   0.2637     -2.8400

Leave-one-state-out range:
6.25% to 9.80%


## Section 4: Does the Finding Hold?

The results broadly support Cheng and Hoekstra's finding that Castle Doctrine expansions are associated with increased homicide rates. The original estimate implies an 8.34% increase, while the alternative group-time DiD estimates exceed 11%. The effect remains positive under alternative weighting, treatment timing, and leave-one-state-out specifications.

However, the unweighted estimate loses statistical significance, and positive pre-treatment event-study coefficients raise concerns about parallel trends. Overall, the finding appears robust in direction, but its exact magnitude and causal interpretation remain uncertain.

In [97]:

# Final report numbers

REPORT_NUMBERS = {
    "Published coefficient": PAPER_NUMBER,
    "Replicated coefficient": ESTIMATE,
    "Replication gap": ESTIMATE - PAPER_NUMBER,
    "Baseline effect (%)": 100 * (np.exp(ESTIMATE) - 1),
    "Baseline p-value": twfe_fit.pvalues["cdl"],
    "Modern DiD never-treated": att_gt(OUTCOME, WEIGHTED, "never")[0],
    "Modern DiD not-yet-treated": att_gt(OUTCOME, WEIGHTED, "notyet")[0],
    "Unweighted effect (%)": 100 * (
        np.exp(unweighted_fit.params["cdl"]) - 1
    ),
    "First-full-year effect (%)": timing_results.loc[
        timing_results["Specification"] == "First full year only",
        "Effect (%)"
    ].iloc[0],
    "Placebo coefficient": placebo_fit.params["fake_cdl"],
    "Placebo p-value": placebo_fit.pvalues["fake_cdl"],
    "Leave-one-out minimum (%)": loo_results["Effect (%)"].min(),
    "Leave-one-out maximum (%)": loo_results["Effect (%)"].max()
}

for name, value in REPORT_NUMBERS.items():
    print(f"{name}: {float(value):.6f}")


Published coefficient: 0.080100
Replicated coefficient: 0.080064
Replication gap: -0.000036
Baseline effect (%): 8.335587
Baseline p-value: 0.025729
Modern DiD never-treated: 0.108177
Modern DiD not-yet-treated: 0.109476
Unweighted effect (%): 9.166209
First-full-year effect (%): 7.845907
Placebo coefficient: -0.028811
Placebo p-value: 0.263743
Leave-one-out minimum (%): 6.251400
Leave-one-out maximum (%): 9.795075


In [98]:

# Final comparison: TWFE vs modern DiD

rows = [{
    "Estimator": "Original TWFE replication",
    "Coefficient": ESTIMATE,
    "Standard Error": twfe_fit.bse["cdl"]
}, {
    "Estimator": "TWFE (state + year FE)",
    "Coefficient": sy.params["cdl"],
    "Standard Error": sy.bse["cdl"]
}]

for control in ["never", "notyet"]:
    estimate, _, _ = att_gt(OUTCOME, WEIGHTED, control)

    se = state_bootstrap(
        OUTCOME, WEIGHTED, control,
        reps=200, seed=42
    )

    rows.append({
        "Estimator": f"Modern DiD ({control}-treated controls)",
        "Coefficient": estimate,
        "Standard Error": se
    })

final_comparison = pd.DataFrame(rows)

final_comparison["Effect (%)"] = 100 * (
    np.exp(final_comparison["Coefficient"]) - 1
)

final_comparison["95% CI Lower (%)"] = 100 * (
    np.exp(
        final_comparison["Coefficient"]
        - 1.96 * final_comparison["Standard Error"]
    ) - 1
)

final_comparison["95% CI Upper (%)"] = 100 * (
    np.exp(
        final_comparison["Coefficient"]
        + 1.96 * final_comparison["Standard Error"]
    ) - 1
)

print("TWFE VS MODERN DiD")
print(final_comparison.round(4).to_string(index=False))

# Examine pre-treatment event-study coefficients
pretrends = table.loc[table.index < 0].copy()

print("\nPRE-TREATMENT EVENT STUDY")
print(pretrends.to_string())

# Export results
final_comparison.to_csv("did_comparison.csv", index=False)
table.to_csv("event_study_results.csv")

print("\nSaved DiD comparison and event-study tables.")


TWFE VS MODERN DiD
                           Estimator  Coefficient  Standard Error  Effect (%)  95% CI Lower (%)  95% CI Upper (%)
           Original TWFE replication       0.0801          0.0359      8.3356            0.9750           16.2328
              TWFE (state + year FE)       0.0801          0.0359      8.3356            0.9750           16.2328
 Modern DiD (never-treated controls)       0.1082          0.0277     11.4245            5.5396           17.6375
Modern DiD (notyet-treated controls)       0.1095          0.0275     11.5694            5.7173           17.7454

PRE-TREATMENT EVENT STUDY
    CS by hand  TWFE event study
-5      0.0622            0.0573
-4      0.0246            0.0317
-3      0.0480            0.0453
-2      0.0362            0.0410

Saved DiD comparison and event-study tables.


In [99]:

# Generate pinned dependencies for reproducibility

from importlib.metadata import version

packages = [
    "numpy",
    "pandas",
    "scipy",
    "statsmodels",
    "matplotlib"
]

with open("requirements.txt", "w") as f:
    for package in packages:
        try:
            f.write(f"{package}=={version(package)}\n")
        except Exception:
            print(f"Package not installed: {package}")

print(open("requirements.txt").read())


numpy==2.4.6
pandas==2.2.3
scipy==1.16.3
statsmodels==0.15.0
matplotlib==3.10.0

